# Module 04.15 — Short URLs (`short-url`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.15 Short URLs (`short-url`)

When you click **Share → Get short URL** in Kibana, a `short-url` object is created that
maps a short hash (the **slug**) to a full application state URL. The browser navigates to
`/goto/<slug>` and Kibana redirects it to the full URL — which can encode dashboard panel
positions, time range, filter state, and other URL parameters that would otherwise produce
an unwieldy link.

Short URLs use the **Locator** system internally: instead of storing the raw URL, the
object stores a `locatorId` (e.g. `DASHBOARD_APP_LOCATOR`) and a `params` object
(e.g. `{dashboardId: "abc123"}`). Kibana resolves the actual URL from this at redirect
time. This makes short URLs more resilient to Kibana upgrades — the locator API can adapt
the params to the new URL structure without breaking every existing short URL.

The restore concern here is the **dangling reference** problem. A short URL points to a
specific dashboard (or Discover session, or map) by ID. If the target object was deleted
and not included in the restore, navigating to the short URL will produce a "not found"
error. Because short URLs are often embedded in external communication (emails, wiki pages,
dashboards in other tools), broken short URLs after a partial restore can be a significant
operational problem. This is one more reason to prefer full feature-state restores over
selective object restores.

In [ ]:
heading('4.15 Short URLs')

# Create a short URL via the API
dashboards = find_saved_objects('dashboard')
if dashboards:
    d = dashboards[0]
    short_url_resp = kibana_post('/api/short_url', {
        'locatorId': 'DASHBOARD_APP_LOCATOR',
        'params': {'dashboardId': d['id']},
    })
    short_id = short_url_resp.get('id')
    slug = short_url_resp.get('slug', '')
    success(f'Short URL created: id={short_id}  slug={slug}')
    pp(short_url_resp, 'short-url object')
    info('The short URL is resolved by Kibana at /goto/{slug}')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
# Short URLs redirect to the full dashboard URL in the browser.
# The short_url_resp above contains the 'slug' field — construct the redirect:
if 'short_url_resp' in dir() and short_url_resp:
    slug = short_url_resp.get('slug', '')
    kibana_link(f'/goto/{slug}', f'Short URL redirect → /goto/{slug}')

In [ ]:
heading('Short URL — snapshot → delete → restore')

# Ensure short_url_id is set — check if variable exists, otherwise find/create
try:
    short_url_id
except NameError:
    existing = find_saved_objects('short-url')
    if existing:
        short_url_id = existing[0]['id']
        info(f'Reusing existing short URL: id={short_url_id}')
    else:
        dashboards = find_saved_objects('dashboard')
        if not dashboards:
            raise RuntimeError('No dashboards found — run 04_06 first')
        dash_id = dashboards[0]['id']
        su_resp = kibana_post('/api/short_url', {
            'locatorId': 'DASHBOARD_APP_LOCATOR',
            'params': {'dashboardId': dash_id},
        })
        short_url_id = su_resp['id']
        success(f'Short URL created: id={short_url_id}')

obj = kibana_get(f'/api/saved_objects/short-url/{short_url_id}')
pp(obj, 'short-url saved object')
info('Key fields:')
info('  slug      — the short code appended to /goto/')
info('  locatorId — which Kibana app handles the redirect')
info('  params    — full state passed to the locator on redirect')

if not any(o['id'] == short_url_id for o in find_saved_objects('short-url')):
    warn('Short URL missing — recreating before snapshot')
    dash_id = find_saved_objects('dashboard')[0]['id']
    su_resp = kibana_post('/api/short_url', {'locatorId': 'DASHBOARD_APP_LOCATOR', 'params': {'dashboardId': dash_id}})
    short_url_id = su_resp['id']

snap_delete_restore_cycle('snap-shorturl-test', 'Short URL')

kibana_delete(f'/api/saved_objects/short-url/{short_url_id}')
warn(f'Accidentally deleted short URL {short_url_id}')
assert not any(o['id'] == short_url_id for o in find_saved_objects('short-url')), 'Should be gone'

restore_kibana_state(es, SNAP_REPO, 'snap-shorturl-test')
time.sleep(3)

restored = find_saved_objects('short-url')
assert any(o['id'] == short_url_id for o in restored), 'Short URL should be restored'
success(f'Short URL {short_url_id} successfully restored!')